In [5]:
# Install if needed:
# pip install yfinance pandas numpy requests

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# -----------------------------
# 1. Define portfolio universe
# -----------------------------

stocks = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "JPM", "XOM", "JNJ"
]

mutual_funds = [
    "VFIAX", "VTSAX", "VBTLX", "VIMAX", "VSMAX",
    "VGSLX", "VTIAX"
]

other_assets = [
    "GLD",   # Gold ETF
    "SLV",   # Silver ETF
    "USO",   # Oil ETF
    "TLT",   # Long-term Treasury ETF
    "BND",   # Bond ETF
    "BTC-USD",
    "ETH-USD"
]

tickers = stocks + mutual_funds + other_assets


# -----------------------------
# 2. Download historical prices
# -----------------------------

def get_price_data(tickers, start_date="2020-01-01", end_date=None):
    if end_date is None:
        end_date = datetime.today().strftime("%Y-%m-%d")

    data = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        group_by="ticker",
        progress=False
    )

    return data


price_data = get_price_data(tickers)


# -----------------------------
# 3. Create adjusted close table
# -----------------------------

def extract_close_prices(price_data, tickers):
    close_prices = pd.DataFrame()

    for ticker in tickers:
        try:
            close_prices[ticker] = price_data[ticker]["Close"]
        except Exception:
            print(f"Close price not found for {ticker}")

    return close_prices


close_prices = extract_close_prices(price_data, tickers)


# -----------------------------
# 4. Calculate returns
# -----------------------------

daily_returns = close_prices.pct_change()
monthly_prices = close_prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change()


# -----------------------------
# 5. Technical indicators
# -----------------------------

def calculate_technical_indicators(close_prices):
    technicals = {}

    for ticker in close_prices.columns:
        df = pd.DataFrame()
        df["close"] = close_prices[ticker]

        # IMPORTANT: remove non-trading days for each ticker before calculations
        df = df.dropna(subset=["close"]).copy()

        df["return_1m"] = df["close"].pct_change(21)
        df["return_3m"] = df["close"].pct_change(63)
        df["return_6m"] = df["close"].pct_change(126)
        df["return_12m"] = df["close"].pct_change(252)

        df["volatility_3m"] = df["close"].pct_change().rolling(63).std() * np.sqrt(252)
        df["moving_avg_50"] = df["close"].rolling(50).mean()
        df["moving_avg_200"] = df["close"].rolling(200).mean()

        df["trend_signal"] = np.where(
            df["moving_avg_50"] > df["moving_avg_200"], 1, 0
        )

        technicals[ticker] = df

    return technicals

technical_data = calculate_technical_indicators(close_prices)


technical_snapshot = []
skipped_tickers = []

for ticker, df in technical_data.items():

    # Only remove missing close price for this ticker
    clean_df = df.dropna(subset=["close"]).copy()

    # Need enough observations for 12M return and 200-day MA
    if len(clean_df) < 252:
        skipped_tickers.append(ticker)
        continue

    # Now remove rows where calculated indicators are missing
    clean_df = clean_df.dropna(subset=[
        "return_1m",
        "return_3m",
        "return_6m",
        "return_12m",
        "volatility_3m",
        "moving_avg_50",
        "moving_avg_200"
    ])

    if clean_df.empty:
        skipped_tickers.append(ticker)
        continue

    latest = clean_df.iloc[-1]

    technical_snapshot.append({
        "Ticker": ticker,
        "1M Return": latest["return_1m"],
        "3M Return": latest["return_3m"],
        "6M Return": latest["return_6m"],
        "12M Return": latest["return_12m"],
        "3M Volatility": latest["volatility_3m"],
        "Trend Signal": latest["trend_signal"]
    })

technical_snapshot = pd.DataFrame(technical_snapshot)

print("Technical snapshot created successfully.")
print(technical_snapshot)

if skipped_tickers:
    print("Skipped tickers:")
    print(skipped_tickers)

# -----------------------------
# 6. Gather basic fundamentals
# -----------------------------

def get_fundamental_data(tickers):
    rows = []

    for ticker in tickers:
        try:
            asset = yf.Ticker(ticker)
            info = asset.info

            rows.append({
                "Ticker": ticker,
                "Name": info.get("shortName"),
                "Sector": info.get("sector"),
                "Industry": info.get("industry"),
                "Market Cap": info.get("marketCap"),
                "Trailing PE": info.get("trailingPE"),
                "Forward PE": info.get("forwardPE"),
                "Price to Book": info.get("priceToBook"),
                "Profit Margin": info.get("profitMargins"),
                "Revenue Growth": info.get("revenueGrowth"),
                "Earnings Growth": info.get("earningsGrowth"),
                "Debt to Equity": info.get("debtToEquity"),
                "Dividend Yield": info.get("dividendYield"),
                "Beta": info.get("beta")
            })

        except Exception as e:
            print(f"Could not fetch fundamentals for {ticker}: {e}")

    return pd.DataFrame(rows)


fundamental_data = get_fundamental_data(tickers)


# -----------------------------
# 7. Gather recent news headlines
# -----------------------------

def get_news_data(tickers, max_news=5):
    news_rows = []

    for ticker in tickers:
        try:
            asset = yf.Ticker(ticker)
            news = asset.news

            for item in news[:max_news]:
                news_rows.append({
                    "Ticker": ticker,
                    "Title": item.get("title"),
                    "Publisher": item.get("publisher"),
                    "Link": item.get("link"),
                    "Published Date": datetime.fromtimestamp(
                        item.get("providerPublishTime")
                    ) if item.get("providerPublishTime") else None
                })

        except Exception as e:
            print(f"Could not fetch news for {ticker}: {e}")

    return pd.DataFrame(news_rows)


news_data = get_news_data(tickers)


# -----------------------------
# 8. Save outputs
# -----------------------------

close_prices.to_csv("close_prices.csv")
daily_returns.to_csv("daily_returns.csv")
monthly_returns.to_csv("monthly_returns.csv")
technical_snapshot.to_csv("technical_snapshot.csv", index=False)
fundamental_data.to_csv("fundamental_data.csv", index=False)
news_data.to_csv("news_data.csv", index=False)


print("Data collection complete.")
print("Files created:")
print("- close_prices.csv")
print("- daily_returns.csv")
print("- monthly_returns.csv")
print("- technical_snapshot.csv")
print("- fundamental_data.csv")
print("- news_data.csv")

Technical snapshot created successfully.
     Ticker  1M Return  3M Return  6M Return  12M Return  3M Volatility  \
0      AAPL   0.095881   0.080631   0.040690    0.324052       0.265240   
1      MSFT   0.122019  -0.034637  -0.231533    0.056552       0.300002   
2      NVDA   0.129161   0.038355  -0.041384    0.822409       0.391256   
3      AMZN   0.273971   0.121020   0.164829    0.454615       0.330100   
4     GOOGL   0.296916   0.141898   0.406614    1.437200       0.346447   
5      META   0.050964  -0.149656  -0.188782    0.112293       0.405011   
6      TSLA   0.025075  -0.091982  -0.153171    0.385101       0.380897   
7       JPM   0.063271   0.026739   0.032653    0.302211       0.239802   
8       XOM  -0.049944   0.087469   0.332015    0.495772       0.297877   
9       JNJ  -0.069351   0.005051   0.231768    0.493954       0.165141   
10    VFIAX   0.100211   0.044836   0.055441    0.313817       0.152326   
11    VTSAX   0.099015   0.046365   0.058695    0.316947   

In [6]:
master_df = technical_snapshot.merge(
    fundamental_data,
    on="Ticker",
    how="left"
)

master_df.to_csv("master_asset_data.csv", index=False)

print(master_df.head())

  Ticker  1M Return  3M Return  6M Return  12M Return  3M Volatility  \
0   AAPL   0.095881   0.080631   0.040690    0.324052       0.265240   
1   MSFT   0.122019  -0.034637  -0.231533    0.056552       0.300002   
2   NVDA   0.129161   0.038355  -0.041384    0.822409       0.391256   
3   AMZN   0.273971   0.121020   0.164829    0.454615       0.330100   
4  GOOGL   0.296916   0.141898   0.406614    1.437200       0.346447   

   Trend Signal                   Name                  Sector  \
0           1.0             Apple Inc.              Technology   
1           0.0  Microsoft Corporation              Technology   
2           1.0     NVIDIA Corporation              Technology   
3           0.0       Amazon.com, Inc.       Consumer Cyclical   
4           1.0          Alphabet Inc.  Communication Services   

                         Industry    Market Cap  Trailing PE  Forward PE  \
0            Consumer Electronics  4.109006e+12    33.915257   29.386591   
1       Software -

In [ ]:
stocks = ["AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","JPM","XOM","JNJ"]
mutual_funds = ["VFIAX","VTSAX","VBTLX","VIMAX","VSMAX","VGSLX","VTIAX"]
other_assets = ["GLD","SLV","USO","TLT","BND","BTC-USD","ETH-USD"]

def classify_asset(ticker):
    if ticker in stocks:
        return "Stock"
    elif ticker in mutual_funds:
        return "Mutual Fund"
    else:
        return "Other Asset"

master_df["Asset Type"] = master_df["Ticker"].apply(classify_asset)



def normalize(series, reverse=False):
    series = pd.to_numeric(series, errors="coerce")
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)

    if reverse:
        return (max_val - series) / (max_val - min_val)
    else:
        return (series - min_val) / (max_val - min_val)

In [ ]:
# Business Strength Score = 30% Profitability + 30% Growth + 20% Valuation + 20% Stability (stocks only)
df = master_df.copy()

stock_mask = df["Asset Type"] == "Stock"

df.loc[stock_mask, "profit_score"] = normalize(df.loc[stock_mask, "Profit Margin"])
df.loc[stock_mask, "growth_score"] = (
    normalize(df.loc[stock_mask, "Revenue Growth"]) +
    normalize(df.loc[stock_mask, "Earnings Growth"])
) / 2

df.loc[stock_mask, "valuation_score"] = (
    normalize(df.loc[stock_mask, "Trailing PE"], reverse=True) +
    normalize(df.loc[stock_mask, "Price to Book"], reverse=True)
) / 2

df.loc[stock_mask, "stability_score"] = (
    normalize(df.loc[stock_mask, "Debt to Equity"], reverse=True) +
    normalize(df.loc[stock_mask, "Beta"], reverse=True)
) / 2

df.loc[stock_mask, "Reality Score"] = (
    0.30 * df.loc[stock_mask, "profit_score"] +
    0.30 * df.loc[stock_mask, "growth_score"] +
    0.20 * df.loc[stock_mask, "valuation_score"] +
    0.20 * df.loc[stock_mask, "stability_score"]
)

In [ ]:
# Price score = 20% Momentum + 20% Trend + 20% Volatility + 40% (for future: sentiment, news, etc.) (foe all assets)
df["momentum_score"] = (
    normalize(df["1M Return"]) * 0.20 +
    normalize(df["3M Return"]) * 0.30 +
    normalize(df["6M Return"]) * 0.30 +
    normalize(df["12M Return"]) * 0.20
)

df["volatility_score"] = normalize(df["3M Volatility"], reverse=True)

df["Price Score"] = (
    0.65 * df["momentum_score"] +
    0.20 * df["Trend Signal"] +
    0.15 * df["volatility_score"]
)

In [ ]:
# Dummy for sentiment and news scores - in real implementation, would replace with sentiment analysis of news headlines and social media
df["Mispricing Gap"] = df["Reality Score"] - df["Price Score"]

def recommendation(row):
    if row["Asset Type"] != "Stock":
        if row["Price Score"] >= 0.65:
            return "Increase / Keep Strong"
        elif row["Price Score"] <= 0.35:
            return "Reduce / Watch"
        else:
            return "Hold"
    
    if row["Mispricing Gap"] >= 0.15:
        return "Increase"
    elif row["Mispricing Gap"] <= -0.15:
        return "Reduce"
    else:
        return "Hold"

df["Recommendation"] = df.apply(recommendation, axis=1)

df.to_csv("scored_asset_data.csv", index=False)

df[[
    "Ticker", "Asset Type", "Reality Score", "Price Score",
    "Mispricing Gap", "Recommendation"
]]

In [8]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load master file
# -----------------------------

master_df = pd.read_csv("master_asset_data.csv")


# -----------------------------
# 2. Asset classification
# -----------------------------

stocks = ["AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","JPM","XOM","JNJ"]
mutual_funds = ["VFIAX","VTSAX","VBTLX","VIMAX","VSMAX","VGSLX","VTIAX"]
other_assets = ["GLD","SLV","USO","TLT","BND","BTC-USD","ETH-USD"]

def classify_asset(ticker):
    if ticker in stocks:
        return "Stock"
    elif ticker in mutual_funds:
        return "Mutual Fund"
    else:
        return "Other Asset"

df = master_df.copy()
df["Asset Type"] = df["Ticker"].apply(classify_asset)


# -----------------------------
# 3. Normalization function
# -----------------------------

def normalize(series, reverse=False):
    series = pd.to_numeric(series, errors="coerce")
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)

    if reverse:
        return (max_val - series) / (max_val - min_val)
    else:
        return (series - min_val) / (max_val - min_val)


# -----------------------------
# 4. Reality Score — stocks only
# -----------------------------

stock_mask = df["Asset Type"] == "Stock"

df.loc[stock_mask, "profit_score"] = normalize(df.loc[stock_mask, "Profit Margin"])

df.loc[stock_mask, "growth_score"] = (
    normalize(df.loc[stock_mask, "Revenue Growth"]) +
    normalize(df.loc[stock_mask, "Earnings Growth"])
) / 2

df.loc[stock_mask, "valuation_score"] = (
    normalize(df.loc[stock_mask, "Trailing PE"], reverse=True) +
    normalize(df.loc[stock_mask, "Price to Book"], reverse=True)
) / 2

df.loc[stock_mask, "stability_score"] = (
    normalize(df.loc[stock_mask, "Debt to Equity"], reverse=True) +
    normalize(df.loc[stock_mask, "Beta"], reverse=True)
) / 2

df.loc[stock_mask, "Reality Score"] = (
    0.30 * df.loc[stock_mask, "profit_score"] +
    0.30 * df.loc[stock_mask, "growth_score"] +
    0.20 * df.loc[stock_mask, "valuation_score"] +
    0.20 * df.loc[stock_mask, "stability_score"]
)


# -----------------------------
# 5. Price Score — all assets
# -----------------------------

df["momentum_score"] = (
    0.20 * normalize(df["1M Return"]) +
    0.30 * normalize(df["3M Return"]) +
    0.30 * normalize(df["6M Return"]) +
    0.20 * normalize(df["12M Return"])
)

df["volatility_score"] = normalize(df["3M Volatility"], reverse=True)

df["Price Score"] = (
    0.65 * df["momentum_score"] +
    0.20 * df["Trend Signal"] +
    0.15 * df["volatility_score"]
)


# -----------------------------
# 6. Mispricing Gap
# -----------------------------

df["Mispricing Gap"] = df["Reality Score"] - df["Price Score"]


# -----------------------------
# 7. Recommendation logic
# -----------------------------

def recommendation(row):
    if row["Asset Type"] == "Stock":
        if row["Mispricing Gap"] >= 0.15:
            return "Increase"
        elif row["Mispricing Gap"] <= -0.15:
            return "Reduce"
        else:
            return "Hold"

    else:
        if row["Price Score"] >= 0.65:
            return "Increase / Keep Strong"
        elif row["Price Score"] <= 0.35:
            return "Reduce / Watch"
        else:
            return "Hold"


df["Recommendation"] = df.apply(recommendation, axis=1)


# -----------------------------
# 8. Plain-English explanation
# -----------------------------

def explain(row):
    if row["Asset Type"] == "Stock":
        if row["Recommendation"] == "Increase":
            return "Business strength looks better than market price reaction. Possible long-term opportunity."
        elif row["Recommendation"] == "Reduce":
            return "Price movement looks stronger than business support. Possible overreaction risk."
        else:
            return "Business strength and price behavior look reasonably balanced."

    else:
        if row["Recommendation"] == "Increase / Keep Strong":
            return "Asset shows strong price trend with acceptable risk."
        elif row["Recommendation"] == "Reduce / Watch":
            return "Asset shows weak price behavior or higher risk."
        else:
            return "Asset looks stable enough to hold for diversification."


df["Simple Explanation"] = df.apply(explain, axis=1)


# -----------------------------
# 9. Save scored data
# -----------------------------

output_cols = [
    "Ticker", "Asset Type",
    "Reality Score", "Price Score", "Mispricing Gap",
    "Recommendation", "Simple Explanation"
]

scored_df = df[output_cols].copy()

scored_df.to_csv("scored_asset_data.csv", index=False)

print(scored_df)

     Ticker   Asset Type  Reality Score  Price Score  Mispricing Gap  \
0      AAPL        Stock            NaN     0.506292             NaN   
1      MSFT        Stock       0.635119     0.222838        0.412281   
2      NVDA        Stock       0.828801     0.501241        0.327560   
3      AMZN        Stock       0.488652     0.391882        0.096770   
4     GOOGL        Stock       0.697298     0.704428       -0.007130   
5      META        Stock       0.651812     0.161048        0.490765   
6      TSLA        Stock       0.266064     0.192886        0.073178   
7       JPM        Stock            NaN     0.286533             NaN   
8       XOM        Stock       0.451771     0.506607       -0.054836   
9       JNJ        Stock       0.427755     0.496124       -0.068369   
10    VFIAX  Mutual Fund            NaN     0.527256             NaN   
11    VTSAX  Mutual Fund            NaN     0.527082             NaN   
12    VBTLX  Mutual Fund            NaN     0.479437            

In [13]:
import pandas as pd
import numpy as np

# Load full master data, not scored file
df = pd.read_csv("master_asset_data.csv")

# Asset groups
stocks = ["AAPL","MSFT","NVDA","AMZN","GOOGL","META","TSLA","JPM","XOM","JNJ"]
mutual_funds = ["VFIAX","VTSAX","VBTLX","VIMAX","VSMAX","VGSLX","VTIAX"]
other_assets = ["GLD","SLV","USO","TLT","BND","BTC-USD","ETH-USD"]

def classify_asset(ticker):
    if ticker in stocks:
        return "Stock"
    elif ticker in mutual_funds:
        return "Mutual Fund"
    else:
        return "Other Asset"

df["Asset Type"] = df["Ticker"].apply(classify_asset)

# Normalize function
def normalize(series, reverse=False):
    series = pd.to_numeric(series, errors="coerce")
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(0.5, index=series.index)

    if reverse:
        return (max_val - series) / (max_val - min_val)
    else:
        return (series - min_val) / (max_val - min_val)

# Fill missing fundamentals for stocks
fundamental_cols = [
    "Profit Margin", "Revenue Growth", "Earnings Growth",
    "Trailing PE", "Price to Book", "Debt to Equity", "Beta"
]

for col in fundamental_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    stock_median = df.loc[df["Asset Type"] == "Stock", col].median()

    df.loc[df["Asset Type"] == "Stock", col] = (
        df.loc[df["Asset Type"] == "Stock", col].fillna(stock_median)
    )

# Reality Score — stocks only
stock_mask = df["Asset Type"] == "Stock"

df.loc[stock_mask, "profit_score"] = normalize(df.loc[stock_mask, "Profit Margin"])

df.loc[stock_mask, "growth_score"] = (
    normalize(df.loc[stock_mask, "Revenue Growth"]) +
    normalize(df.loc[stock_mask, "Earnings Growth"])
) / 2

df.loc[stock_mask, "valuation_score"] = (
    normalize(df.loc[stock_mask, "Trailing PE"], reverse=True) +
    normalize(df.loc[stock_mask, "Price to Book"], reverse=True)
) / 2

df.loc[stock_mask, "stability_score"] = (
    normalize(df.loc[stock_mask, "Debt to Equity"], reverse=True) +
    normalize(df.loc[stock_mask, "Beta"], reverse=True)
) / 2

df.loc[stock_mask, "Reality Score"] = (
    0.30 * df.loc[stock_mask, "profit_score"] +
    0.30 * df.loc[stock_mask, "growth_score"] +
    0.20 * df.loc[stock_mask, "valuation_score"] +
    0.20 * df.loc[stock_mask, "stability_score"]
)

# Price Score — all assets
df["momentum_score"] = (
    0.20 * normalize(df["1M Return"]) +
    0.30 * normalize(df["3M Return"]) +
    0.30 * normalize(df["6M Return"]) +
    0.20 * normalize(df["12M Return"])
)

df["volatility_score"] = normalize(df["3M Volatility"], reverse=True)

df["Price Score"] = (
    0.65 * df["momentum_score"] +
    0.20 * df["Trend Signal"] +
    0.15 * df["volatility_score"]
)

# Mispricing Gap
df["Mispricing Gap"] = df["Reality Score"] - df["Price Score"]

# Percentile ranks
df["Reality Rank"] = df.groupby("Asset Type")["Reality Score"].rank(pct=True)
df["Price Rank"] = df.groupby("Asset Type")["Price Score"].rank(pct=True)
df["Gap Rank"] = df.groupby("Asset Type")["Mispricing Gap"].rank(pct=True)

# Recommendation
def percentile_recommendation(row):

    if row["Asset Type"] == "Stock":
        if row["Gap Rank"] >= 0.75:
            return "Increase"
        elif row["Gap Rank"] <= 0.25:
            return "Trim / Monitor"
        else:
            return "Hold"

    else:
        if row["Price Rank"] >= 0.75:
            return "Increase / Keep Strong"
        elif row["Price Rank"] <= 0.25:
            return "Reduce / Watch"
        else:
            return "Hold"


df["Smart Recommendation"] = df.apply(percentile_recommendation, axis=1)


# Explanation
def smart_explanation(row):

    if row["Asset Type"] == "Stock":
        if row["Smart Recommendation"] == "Increase":
            return "Business strength ranks better than the market’s current price reaction."
        elif row["Smart Recommendation"] == "Trim / Monitor":
            return "Price has moved faster than business support, so we avoid adding more and monitor closely."
        else:
            return "Business strength and price behavior look fairly balanced."

    else:
        if row["Smart Recommendation"] == "Increase / Keep Strong":
            return "This asset ranks strong within its group based on trend and risk."
        elif row["Smart Recommendation"] == "Reduce / Watch":
            return "This asset ranks weak within its group based on trend and risk."
        else:
            return "This asset sits in the middle range and supports diversification."


df["Smart Explanation"] = df.apply(smart_explanation, axis=1)

# Save
df.to_csv("ranked_asset_data.csv", index=False)

df[[
    "Ticker", "Asset Type",
    "Reality Score", "Price Score", "Mispricing Gap",
    "Reality Rank", "Price Rank", "Gap Rank",
    "Smart Recommendation", "Smart Explanation"
]]

,Ticker,Asset Type,Reality Score,Price Score,Mispricing Gap,Reality Rank,Price Rank,Gap Rank,Smart Recommendation,Smart Explanation
0,AAPL,Stock,0.399710,0.506292,-0.106583,0.2,0.800000,0.1,Trim / Monitor,"Price has moved faster than business support, ..."
1,MSFT,Stock,0.635119,0.222838,0.412281,0.7,0.300000,0.9,Increase,Business strength ranks better than the market...
2,NVDA,Stock,0.828801,0.501241,0.327560,1.0,0.700000,0.8,Increase,Business strength ranks better than the market...
3,AMZN,Stock,0.488652,0.391882,0.096770,0.5,0.500000,0.6,Hold,Business strength and price behavior look fair...
4,GOOGL,Stock,0.697298,0.704428,-0.007130,0.9,1.000000,0.4,Hold,Business strength and price behavior look fair...
5,META,Stock,0.651812,0.161048,0.490765,0.8,0.100000,1.0,Increase,Business strength ranks better than the market...
6,TSLA,Stock,0.266064,0.192886,0.073178,0.1,0.200000,0.5,Hold,Business strength and price behavior look fair...
7,JPM,Stock,0.604444,0.286533,0.317911,0.6,0.400000,0.7,Hold,Business strength and price behavior look fair...
8,XOM,Stock,0.451771,0.506607,-0.054836,0.4,0.900000,0.3,Hold,Business strength and price behavior look fair...
9,JNJ,Stock,0.427755,0.496124,-0.068369,0.3,0.600000,0.2,Trim / Monitor,"Price has moved faster than business support, ..."
